# Geração de tópicos: comparação entre LLMs

Este notebook executa o **mesmo código** de `topic_generation.py`, usando o GPT-6.1 Sol.
Mantenha o notebook, o `.py` e `experiment_config.json` na mesma pasta.
Instale as dependências de `requirements.txt` no kernel e defina `OPENAI_API_KEY` no ambiente.

Lotes, semente, raciocínio, limite de geração, tentativas, prompts, esquema JSON e validação
são compartilhados. Edite `experiment_config.json` para alterar os dois experimentos.


In [ ]:
from pathlib import Path
import importlib
import topic_generation as experiment

# Recarrega o módulo para usar a versão atual do código compartilhado.
experiment = importlib.reload(experiment)


In [ ]:
# Apenas a identificação da LLM, o acesso e o destino dos resultados mudam.
MODEL = "gpt-6.1-sol"
INPUT_FILE = Path("data/geracao.jsonl")
OUTPUT_DIR = Path("outputs/openai")
CONFIG_FILE = Path("experiment_config.json")


In [ ]:
cfg = experiment.parse_args([
    "--provider", "openai",
    "--model", MODEL,
    "--input", str(INPUT_FILE),
    "--output", str(OUTPUT_DIR),
    "--config", str(CONFIG_FILE),
    "--experiment", "both",
])

print("Protocolo compartilhado:", experiment.sha256_json(experiment.protocol_config(cfg)))
print("Parâmetros enviados à LLM:", experiment.generation_parameters(cfg))
report = experiment.run(cfg)
report


Cada execução começa do zero e sobrescreve as saídas dos experimentos selecionados
diretamente em `OUTPUT_DIR`, sem criar pastas `run_`. Outros arquivos são preservados.
As taxonomias original e embaralhada são independentes. Os arquivos por lote servem para auditoria;
nenhum resultado anterior é carregado como checkpoint.

Para comparar os resultados com o vLLM, confira o mesmo `comparison_hash` nos respectivos
`summary.json` de cada ordem. Ele considera o protocolo e a entrada ordenada, sem incluir o modelo.
O `protocol_hash` identifica o procedimento e o `input_hash` identifica os dados na ordem processada.

As duas APIs recebem `reasoning_effort="high"` por padrão, compatível com o Mistral local e o GPT-6.1 Sol.
O significado desse nível e o orçamento
interno de raciocínio dependem de cada LLM. Temperatura e `top_p` são omitidos nas duas chamadas,
pois o GPT-6.1 Sol não aceita esses controles. Isso não garante amostragem nem resultados idênticos.

Tokens, tempo e tentativas são registrados com o mesmo formato. Tokens de raciocínio aparecem
quando o servidor informa esse detalhamento; um valor ausente significa desconhecido.
O limite de geração inclui raciocínio e resposta final. Se houver truncamento, ajuste o limite
compartilhado e a capacidade de contexto do servidor local. Não há estimativa monetária automática.
